# Notebook 3: LSTM-SNP with Fuzzy Gate Replacement

**Dataset**: Dow Jones Industrial Index

## Description
This notebook implements **fuzzy gate replacement** for the LSTM-SNP model. The reset (r), 
consumption (c), and output (o) gates — which normally use hard sigmoid activation — are 
replaced with fuzzy inference systems. Each gate uses 2 Takagi-Sugeno rules with fixed Gaussian 
membership functions and trainable consequent parameters.

The generation gate (a) retains its original tanh activation. The rest of the SNP architecture 
is unchanged. Gradient clipping (norm=1.0) is applied for training stability.

## Theory: Fuzzy Gate Replacement

### Modified Gate Computation
Instead of applying hard sigmoid directly, each gate (r, c, o) uses fuzzy inference:

**Standard**: $r(t) = \sigma(z_r)$ where $z_r = W_r x(t) + U_r u(t-1) + b_r$

**Fuzzy**: $r(t) = FuzzyInference(z_r, \bar{u}(t-1))$

The fuzzy inference uses:

**Fixed Gaussian Membership Functions**:
- $\mu_{low}(z) = \exp\left(-\frac{(z - (-1))^2}{2 \cdot 0.5^2}\right)$
- $\mu_{high}(z) = \exp\left(-\frac{(z - (+1))^2}{2 \cdot 0.5^2}\right)$

**2 Rules per gate** (trainable consequents):
1. IF $z_{gate}$ is low → $y_0 = a_0 z + b_0 \bar{u} + c_0$
2. IF $z_{gate}$ is high → $y_1 = a_1 z + b_1 \bar{u} + c_1$

**Output**: Gate value clipped to $[0, 1]$

### Unchanged
- Generation gate $a(t) = \tanh(\cdot)$ — unchanged
- State update: $u(t) = r \cdot u_{t-1} - c \cdot a$, $h(t) = o \cdot a$ — unchanged
- Gradient clipping (norm ≤ 1.0) applied for stability

## Model Architecture & Implementation

In [1]:
# ============================================================
# ALL IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import Model
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from math import sqrt
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

TensorFlow version: 2.15.0
NumPy version: 1.26.4


### Fuzzy Gate LSTM-SNP Cell

In [2]:
# ============================================================
# Fuzzy Gate LSTM-SNP Cell
# Gates r, c, o use fuzzy inference instead of hard sigmoid.
# Generation gate 'a' keeps tanh (unchanged).
#
# Fixed Gaussian membership functions:
#   μ_low(x)  = exp(-(x - (-1))² / (2·0.5²))
#   μ_high(x) = exp(-(x - (+1))² / (2·0.5²))
#
# Each gate uses 2 Takagi-Sugeno rules with trainable consequents.
# ============================================================

class FuzzyLSTMSNPCell(layers.Layer):
    """
    LSTM-SNP Cell with fuzzy gate replacement.
    Gates r, c, o are computed via fuzzy inference.
    Gate a keeps tanh activation.
    """
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.state_size = units
        self.output_size = units
        self.activation = tf.keras.activations.get('tanh')
        # Fixed membership function parameters
        self.mu_low = -1.0
        self.mu_high = 1.0
        self.sigma = 0.5

    def build(self, input_shape):
        input_dim = input_shape[-1]

        # Standard weights for pre-activation computation
        # 4 gates: r, c, o, a
        self.kernel = self.add_weight(
            shape=(input_dim, self.units * 4),
            initializer='glorot_uniform',
            name='kernel'
        )
        self.recurrent_kernel = self.add_weight(
            shape=(self.units, self.units * 4),
            initializer='orthogonal',
            name='recurrent_kernel'
        )
        self.bias = self.add_weight(
            shape=(self.units * 4,),
            initializer='zeros',
            name='bias'
        )

        # Fuzzy consequent parameters for 3 gates (r, c, o)
        # Each gate has 2 rules, each rule has 3 params (a, b, c_param)
        # rule_i: y_i = a_i * z_gate + b_i * u_mean + c_i
        # Store as instance attributes for reliable access
        self.fuzzy_params = {}
        for gate_name in ['r', 'c', 'o']:
            self.fuzzy_params[gate_name] = {}
            for rule_idx in range(2):
                self.fuzzy_params[gate_name][rule_idx] = {
                    'a': self.add_weight(
                        shape=(self.units,),
                        initializer=tf.keras.initializers.RandomUniform(-0.1, 0.1),
                        name=f'fuzzy_{gate_name}_rule{rule_idx}_a'
                    ),
                    'b': self.add_weight(
                        shape=(self.units,),
                        initializer=tf.keras.initializers.RandomUniform(-0.1, 0.1),
                        name=f'fuzzy_{gate_name}_rule{rule_idx}_b'
                    ),
                    'c_param': self.add_weight(
                        shape=(self.units,),
                        initializer=tf.keras.initializers.Zeros(),
                        name=f'fuzzy_{gate_name}_rule{rule_idx}_c'
                    ),
                }

    def _gaussian_mf(self, x, center):
        """Fixed Gaussian membership function."""
        return tf.exp(-tf.square(x - center) / (2.0 * self.sigma ** 2))

    def _fuzzy_gate(self, z_gate, u_mean, gate_name):
        """Compute gate value via fuzzy inference (2 rules)."""
        # Membership degrees (fixed)
        w_low = self._gaussian_mf(z_gate, self.mu_low)
        w_high = self._gaussian_mf(z_gate, self.mu_high)

        # Get trainable consequent parameters from instance dict
        p = self.fuzzy_params[gate_name]

        # Rule outputs (trainable linear consequents)
        y0 = p[0]['a'] * z_gate + p[0]['b'] * u_mean + p[0]['c_param']
        y1 = p[1]['a'] * z_gate + p[1]['b'] * u_mean + p[1]['c_param']

        # Weighted average defuzzification
        numerator = w_low * y0 + w_high * y1
        denominator = w_low + w_high + 1e-8
        output = numerator / denominator

        # Clip to [0, 1] since gates should be bounded
        return tf.clip_by_value(output, 0.0, 1.0)

    def call(self, inputs, states):
        u_tm1 = states[0]

        z = tf.matmul(inputs, self.kernel) + \
            tf.matmul(u_tm1, self.recurrent_kernel) + self.bias

        z0 = z[:, :self.units]
        z1 = z[:, self.units:2*self.units]
        z2 = z[:, 2*self.units:3*self.units]
        z3 = z[:, 3*self.units:]

        # Mean of previous state for fuzzy rule input
        u_mean = tf.reduce_mean(u_tm1, axis=-1, keepdims=True)
        u_mean = tf.tile(u_mean, [1, self.units])

        # Fuzzy gates
        r = self._fuzzy_gate(z0, u_mean, 'r')
        c = self._fuzzy_gate(z1, u_mean, 'c')
        o = self._fuzzy_gate(z2, u_mean, 'o')
        a = self.activation(z3)  # tanh unchanged

        u = r * u_tm1 - c * a
        h = o * a

        return h, [u]

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units})
        return config

### Build Model

In [3]:
# ============================================================
# Model Construction: LSTM-SNP with Fuzzy Gate Replacement
# Uses FuzzyLSTMSNPCell instead of LSTMSNPCell.
# Gradient clipping applied for stability.
# ============================================================

def build_model(input_dim, units, batch_size):
    cell = FuzzyLSTMSNPCell(units)
    rnn = layers.RNN(cell, return_sequences=False, stateful=True)

    inputs = tf.keras.Input(batch_shape=(batch_size, 1, input_dim))
    x = rnn(inputs)
    outputs = layers.Dense(1)(x)

    model = tf.keras.Model(inputs, outputs)
    optimizer = tf.keras.optimizers.legacy.Adam(clipnorm=1.0)
    model.compile(optimizer=optimizer, loss='mean_squared_error')
    return model

In [4]:
# Quick model check
model = build_model(input_dim=1, units=8, batch_size=1)
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(1, 1, 1)]               0         
                                                                 
 rnn (RNN)                   (1, 8)                    464       
                                                                 
 dense (Dense)               (1, 1)                    9         
                                                                 
Total params: 473 (1.85 KB)
Trainable params: 473 (1.85 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


## Data Pipeline — Dow Jones Industrial Index

In [5]:
# ============================================================
# 1. Load Time Series Data
# ============================================================

series = pd.read_csv(
    '../../content/monthly-closings-of-the-dowjones.csv',
    header=0,
    parse_dates=[0],
    index_col=0
)

raw_values = series.values.flatten()

import numpy as np
original_raw_values = np.copy(raw_values)
s_x = np.std(original_raw_values)
noise_levels = [0.005]

for lam in noise_levels:
    sigma = lam * s_x
    print("\n" + "="*80)
    print(f"EVALUATING NOISE LEVEL: {lam*100:.1f}% (lambda={lam}, sigma={sigma:.6f})")
    print("="*80 + "\n")
    
    np.random.seed(42)
    tf.random.set_seed(42)
    noise = np.random.normal(0, sigma, size=original_raw_values.shape)
    raw_values = original_raw_values + noise
    
    print(f"Data shape: {raw_values.shape}")
    print(f"First 5 values: {raw_values[:5]}")
    
    # ============================================================
    # 2. First-Order Differencing
    # ============================================================
    
    def difference(dataset, interval=1):
        diff = []
        for i in range(interval, len(dataset)):
            value = dataset[i] - dataset[i - interval]
            diff.append(value)
        return np.array(diff)
    
    diff_values = difference(raw_values, 1)
    
    # ============================================================
    # 3. Convert to Supervised Learning Format (lag=1)
    # ============================================================
    
    def timeseries_to_supervised(data, lag=1):
        df = pd.DataFrame(data)
        columns = [df.shift(i) for i in range(1, lag+1)]
        columns.append(df)
        df = pd.concat(columns, axis=1)
        df.fillna(0, inplace=True)
        return df.values
    
    supervised = timeseries_to_supervised(diff_values, 1)
    print(f"Supervised data shape: {supervised.shape}")
    
    # ============================================================
    # 4. Train-Test Split
    # ============================================================
    
    train, test = supervised[:-60], supervised[-60:]
    print(f"Train: {train.shape}, Test: {test.shape}")
    
    # ============================================================
    # 5. Feature Scaling
    # ============================================================
    
    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaler.fit(train)
    
    train_scaled = scaler.transform(train)
    test_scaled = scaler.transform(test)
    
    # ============================================================
    # 6. Reshape for RNN Input
    # ============================================================
    
    X_train, y_train = train_scaled[:, 0:-1], train_scaled[:, -1]
    X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
    
    X_test, y_test = test_scaled[:, 0:-1], test_scaled[:, -1]
    print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
    
    # ============================================================
    # 30-Run Experiment Protocol
    # ============================================================
    
    all_rmse = []
    all_mse = []
    all_nmse = []
    all_predictions = []
    all_losses = []
    
    for run in range(30):
        print(f'\n===== RUN {run+1}/30 =====')
    
        np.random.seed(run)
        tf.random.set_seed(run)
    
        tf.keras.backend.clear_session()
        model = build_model(input_dim=1, units=8, batch_size=1)
    
        rnn_layer = model.layers[1]
    
        # Training with manual epoch loop + reset_states
        run_losses = []
        for epoch in range(100):
            history = model.fit(
                X_train, y_train,
                epochs=1, batch_size=1,
                verbose=0, shuffle=False
            )
            run_losses.append(history.history['loss'][0])
            rnn_layer.reset_states()
    
        all_losses.append(run_losses)
        print(f'Training complete for run {run+1}')
    
        # Warm-up: condition hidden states on training data
        train_reshaped = train_scaled[:, 0].reshape(len(train_scaled), 1, 1)
        model.predict(train_reshaped, batch_size=1, verbose=0)
    
        # Test predictions (single-step)
        predictions = []
        for i in range(len(test_scaled)):
            X, y = test_scaled[i, 0:-1], test_scaled[i, -1]
            X_input = X.reshape(1, 1, len(X))
            yhat = model.predict(X_input, batch_size=1, verbose=0)[0, 0]
    
            # Invert scaling
            new_row = [x for x in X] + [yhat]
            array = np.array(new_row).reshape(1, len(new_row))
            inverted = scaler.inverse_transform(array)[0, -1]
    
            # Invert differencing
            inverted = inverted + raw_values[len(train) + i]
            predictions.append(inverted)
    
            expected = raw_values[len(train) + i + 1]
            print(f'Month={i+1}, Predicted={inverted:.4f}, Expected={expected:.4f}')
    
        # Compute metrics
        actual = raw_values[-60:]
        rmse = sqrt(mean_squared_error(actual, predictions))
        mse = mean_squared_error(actual, predictions)
        meanV = np.mean(actual)
        dominator = np.linalg.norm(np.array(predictions) - meanV, 2)
        nmse = mse / np.power(dominator, 2)
    
        all_rmse.append(rmse)
        all_mse.append(mse)
        all_nmse.append(nmse)
        all_predictions.append(predictions)
    
        print(f'Run {run+1} — RMSE: {rmse:.6f}, MSE: {mse:.6f}, NMSE: {nmse:.10f}')
    
    # ============================================================
    # Summary Statistics (30 runs)
    # ============================================================
    
    print('\n===== FINAL RESULTS — Fuzzy Gate Replacement on Dow Jones Industrial Index (30 runs) =====')
    print(f'RMSE: {np.mean(all_rmse):.6f} ± {np.std(all_rmse):.6f}')
    print(f'MSE:  {np.mean(all_mse):.6f} ± {np.std(all_mse):.6f}')
    print(f'NMSE: {np.mean(all_nmse):.10f} ± {np.std(all_nmse):.10f}')
    
    best_idx = np.argmin(all_rmse)
    print(f'\nBest run: {best_idx+1}')
    print(f'  RMSE: {all_rmse[best_idx]:.6f}')
    print(f'  MSE:  {all_mse[best_idx]:.6f}')
    print(f'  NMSE: {all_nmse[best_idx]:.10f}')
    
    # ============================================================
    # Predictions vs Actual (Best Run)
    # ============================================================
    
    actual = raw_values[-60:]
    best_predictions = all_predictions[best_idx]
    
    plt.figure(figsize=(12, 5))
    plt.plot(actual, label='Actual', color='blue', linewidth=1.5)
    plt.plot(best_predictions, label='Predicted (Best Run)', color='red',
             linewidth=1.5, linestyle='--')
    plt.title('Fuzzy Gate Replacement — Dow Jones Industrial Index\nPredictions vs Actual (Best of 30 runs)')
    plt.xlabel('Time Step')
    plt.ylabel('Value')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # ============================================================
    # Loss Curve (Best Run)
    # ============================================================
    
    plt.figure(figsize=(12, 4))
    plt.plot(all_losses[best_idx], color='green', linewidth=1.0)
    plt.title('Fuzzy Gate Replacement — Dow Jones Industrial Index\nTraining Loss (Best Run)')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # ============================================================
    # Final Metrics Summary
    # ============================================================
    
    print('=== Best Run Metrics ===')
    print(f'RMSE: {all_rmse[best_idx]:.6f}')
    print(f'MSE:  {all_mse[best_idx]:.6f}')
    print(f'NMSE: {all_nmse[best_idx]:.10f}')
    



EVALUATING NOISE LEVEL: 0.1% (lambda=0.001, sigma=0.105286)

Data shape: (291,)
First 5 values: [3645.05229685 3625.98544276 3634.06819227 3620.66035311 3606.97534702]
Supervised data shape: (290, 2)
Train: (230, 2), Test: (60, 2)
X_train shape: (230, 1, 1), y_train shape: (230,)

===== RUN 1/30 =====
Training complete for run 1
Month=1, Predicted=3795.3461, Expected=3796.0228
Month=2, Predicted=3797.8467, Expected=3793.0048
Month=3, Predicted=3794.5436, Expected=3765.9314
Month=4, Predicted=3761.7606, Expected=3747.2257
Month=5, Predicted=3744.6804, Expected=3754.0667
Month=6, Predicted=3757.6083, Expected=3755.7868
Month=7, Predicted=3758.3865, Expected=3767.0196
Month=8, Predicted=3771.2535, Expected=3750.9303
Month=9, Predicted=3749.0230, Expected=3769.0897
Month=10, Predicted=3771.5604, Expected=3759.9166
Month=11, Predicted=3759.8801, Expected=3784.9879
Month=12, Predicted=3785.2923, Expected=3776.0532
Month=13, Predicted=3776.0805, Expected=3755.0912
Month=14, Predicted=3752.01

KeyboardInterrupt: 

## Training Loop

## Results

## Observations

### Fuzzy Gate Replacement on Dow Jones Industrial Index

**Run the notebook to generate results and fill in observations:**

1. **Prediction Quality**: Compare RMSE/MSE/NMSE with other variants
2. **Training Stability**: Examine loss curves for convergence behavior
3. **Prediction Tracking**: Assess how well predictions track actual values
4. **Computational Cost**: Note training time per run

*After running all 5 variant notebooks, perform cross-variant comparison to evaluate 
whether fuzzy logic improves nonlinearity handling, interpretability, and prediction performance.*